In [1]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_large_table, generate_small_lookup, generate_skewed_table
import time

In [3]:
print("=" * 60)
print("FEATURE 1: Auto Coalesce Shuffle Partitions")
print("=" * 60)

spark_no_aqe = get_spark("Case09_NoAQE", {
    "spark.sql.adaptive.enabled": "false",
    "spark.sql.shuffle.partitions": "200"
})
df=generate_large_table(spark_no_aqe,num_rows=1_000_000)
start=time.time()
result=df.groupBy("category").agg(F.sum("download_bytes"))
result.write.mode("overwrite").parquet("/tmp/case09_no_aqe")
print(f" Without AQE: {time.time()- start:.1f}s (200 shuffle partitions)")
spark_no_aqe.stop()

FEATURE 1: Auto Coalesce Shuffle Partitions


26/04/16 11:17:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/04/16 11:17:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/16 11:17:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/16 11:17:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/16 11:17:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/16 11:17:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/16 11:17:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap me

 Without AQE: 4.3s (200 shuffle partitions)


In [5]:
spark_age=get_spark(
    "Case09_WithAQE",{
        "spark.sql.adaptive.enabled":"true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
        "spark.sql.shuffle.partitions": "200",
        "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64MB"
    }
)

df = generate_large_table(spark_age, num_rows=1_000_000)

start=time.time()
result = df.groupBy("category").agg(F.sum("download_bytes"))
result.write.mode("overwrite").parquet("/tmp/case09_with_aqe")
print(f" With AQE:    {time.time() - start:.1f}s (auto-coalesced)")

 With AQE:    1.3s (auto-coalesced)


# Dynamic Join Strategy Switch

In [7]:
print("\n" + "=" * 60)
print("FEATURE 2: Dynamic Join Strategy Switch")
print("=" * 60)
small=df.filter(F.col("category")=="cat_1")

result2=df.join(small,"account_number")
result2.explain(True)


FEATURE 2: Dynamic Join Strategy Switch
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [account_number])
:- Project [id#55L, account_number#57, event_date#60, session_duration#64, download_bytes#69L, upload_bytes#75L, concat(cat_, cast((id#55L % cast(50 as bigint)) as string)) AS category#82]
:  +- Project [id#55L, account_number#57, event_date#60, session_duration#64, download_bytes#69L, cast((rand(-1161141315268676678) * cast(500000 as double)) as bigint) AS upload_bytes#75L]
:     +- Project [id#55L, account_number#57, event_date#60, session_duration#64, cast((rand(8727436141807137241) * cast(1000000 as double)) as bigint) AS download_bytes#69L]
:        +- Project [id#55L, account_number#57, event_date#60, cast((rand(-6041345643368436727) * cast(3600 as double)) as int) AS session_duration#64]
:           +- Project [id#55L, account_number#57, date_add(cast(2025-01-01 as date), cast((id#55L % cast(90 as bigint)) as int)) AS event_date#60]
:              +- Project [id#55L, cast(

In [12]:
print("\n" + "=" * 60)
print("FEATURE 3: Skew Join Optimization")
print("=" * 60)

spark_age.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark_age.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "64MB")

skewed = generate_skewed_table(spark_age, num_rows=5_000_000, hot_keys=3, hot_key_ratio=0.7)
other = skewed.groupBy("account_number").agg(F.avg("value").alias("avg_val"))

start = time.time()
result3 = skewed.join(other, "account_number")
result3.write.mode("overwrite").parquet("/tmp/case09_skew_aqe")
print(f" AQE Skew Join: {time.time() - start:.1f}s")

spark_age.stop()


FEATURE 3: Skew Join Optimization


26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/16 12:21:40 WARN MemoryManager: Total allocation exceeds 95.00% 

 AQE Skew Join: 2.5s
